**Pipeline overview:**
1. Load pre-engineered features from dataset vn_stocks_master_features.csv and vnindex_ohlcv.csv
2. Chronological 70 / 10 / 20 train–val–test split (NO shuffling)
3. `RobustScaler` fitted on training data only (no leakage)
4. Sliding-window sequence construction
5. Model training with `EarlyStopping` + `ReduceLROnPlateau` with loss function such as Huber loss function.
6. Evaluation: MAE, RMSE, MAPE — plus per-horizon degradation analysis

---
## Section 1 - Imports & Configuration

In [ ]:
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
import matplotlib.dates as mdates
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
import joblib

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
tf.random.set_seed(42)
np.random.seed(42)

# ── Project root logic (Tối ưu cho cả Local và Colab) ────────────────────────
try:
    from google.colab import drive
    # Nếu chạy trên Colab và đã mount Drive
    ROOT = Path('/content/drive/MyDrive/DL4AI-240166-project-1')
    print("✓ Running on Google Colab")
except ImportError:
    # Nếu chạy trên máy cá nhân
    ROOT = Path.cwd().parent
    print("✓ Running Locally")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'Project root: {ROOT}')

---
### CONFIGURATIONS

In [ ]:
CONFIG = {
    # ── Sequence construction ─────────────────────────────────────────────────
    'window_size'  : 20,    # look-back window: 2 trading days ≈ 1 calendar month
                            # Compared to 60 look-back window
    'n_features'   : 18,    # overwritten below after FEATURE_COLS is defined

    # ── Data split ratios ─────────────────────────────────────────────────────
    'train_ratio'  : 0.70,  # 70% of chronological data for training
    'val_ratio'    : 0.10,  # 10% for validation / hyperparameter tuning
    'test_ratio'   : 0.20,  # 20% held out for final evaluation

    # ── Training hyperparameters ──────────────────────────────────────────────
    'batch_size'   : 64,    # mini-batch size for gradient descent
    'epochs'       : 100,   # upper bound (EarlyStopping cuts this short)
    'learning_rate': 1e-3,  # Adam initial learning rate
    'patience'     : 10,    # EarlyStopping: max epochs without val_loss improvement

    # ── Dataset quality filter ────────────────────────────────────────────────
    'min_history'  : 120,   # discard tickers with fewer than 120 trading days

    # ── Forecasting horizons ──────────────────────────────────────────────────
    'n_day'        : 3,     # Task 2.2: predict the price n trading days ahead
    'k_days'       : 5,     # Task 2.3: predict the next k consecutive days
}

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k:<18} = {v}')

### CONFIG Parameter Guide

| Parameter | Value | Rationale |
|---|---|---|
| `window_size` | 20 | 20 trading days ≈ 1 calendar months. 20 trading days ≈ 1 calendar month. More suitable for the short-term volatility characteristics of the VN-Index compared to a 60-day window. |
| `n_features` | 18 | Set dynamically from `FEATURE_COLS` after Section 3. The spec noted 13; actual pipeline produces 18 (6 OHLCV + 12 indicators). |
| `train_ratio` | 0.70 | 70/10/20 split for time-series. More training data reduces variance. |
| `val_ratio` | 0.10 | Drives EarlyStopping and ReduceLROnPlateau decisions. |
| `test_ratio` | 0.20 | **Never seen during training or hyperparameter search.** |
| `batch_size` | 64 | Balances training speed and gradient estimate quality. |
| `epochs` | 100 | Upper bound; EarlyStopping with `patience=10` terminates well before this. |
| `learning_rate` | 1e-3 | Adam default. `ReduceLROnPlateau` halves this if val_loss stagnates for 5 epochs. |
| `patience` | 10 | EarlyStopping: wait 10 epochs without improvement before terminating. |
| `min_history` | 120 | Filters tickers with insufficient history. |
| `n_day` | 3 | Task 2.2: primary evaluation at n=3; degradation also tested at n=1..7. |
| `k_days` | 5 | Task 2.3: one full trading week of consecutive predictions. |

---
## Section 2 - Data Loading & Ticker Filtering

In [ ]:
# ── Data paths ───────────────────────────────────────────────────────────────
DATA_DIR    = ROOT / 'notebooks' / 'data' / 'vietnam'
MASTER_PATH = DATA_DIR / 'vn_stocks_master_features.csv'
VNI_PATH    = DATA_DIR / 'csv' / 'vnindex_ohlcv.csv'

assert MASTER_PATH.exists(), f'File not found: {MASTER_PATH}'
assert VNI_PATH.exists(),    f'File not found: {VNI_PATH}'

# ── Load master features (10 tickers) ────────────────────────────────────────
df_all = pd.read_csv(MASTER_PATH)
df_all['date'] = pd.to_datetime(df_all['date']).dt.normalize()
df_all = df_all.sort_values(['ticker', 'date']).reset_index(drop=True)

# ── Load VN-Index OHLCV ───────────────────────────────────────────────────────
df_vni_raw = pd.read_csv(VNI_PATH)
df_vni_raw['date'] = pd.to_datetime(df_vni_raw['date']).dt.normalize()
df_vni_raw = df_vni_raw.sort_values('date').reset_index(drop=True)

# ── VN-Index feature engineering ──────────────────────────────────────────────
def compute_vni_features(df):
    df = df.copy().sort_values('date').reset_index(drop=True)
    close, high, low = df['close'], df['high'], df['low']

    df['vni_log_return'] = np.log(close / close.shift(1))
    df['vni_ema_10'] = close.ewm(span=10, adjust=False).mean()
    df['vni_ema_20'] = close.ewm(span=20, adjust=False).mean()
    df['vni_ema_50'] = close.ewm(span=50, adjust=False).mean()

    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    df['vni_rsi'] = 100 - (100 / (1 + rs))

    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    df['vni_macd']        = ema_12 - ema_26
    df['vni_macd_signal'] = df['vni_macd'].ewm(span=9, adjust=False).mean()
    df['vni_macd_hist']   = df['vni_macd'] - df['vni_macd_signal']

    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    df['vni_atr'] = tr.rolling(14).mean()

    sma_20 = close.rolling(20).mean()
    std_20 = close.rolling(20).std()
    df['vni_bb_middle'] = sma_20
    df['vni_bb_upper']  = sma_20 + 2 * std_20
    df['vni_bb_lower']  = sma_20 - 2 * std_20

    vni_cols = [c for c in df.columns if c.startswith('vni_')]
    df[vni_cols] = df[vni_cols].fillna(0)
    return df[['date'] + vni_cols]

df_vni = compute_vni_features(df_vni_raw)

print(f'Master loaded : {df_all.shape[0]:,} rows | Tickers: {sorted(df_all["ticker"].unique())}')
print(f'VNI features  : {df_vni.shape[1] - 1} indicators')

---


In [ ]:
# ── Filter by minimum history ─────────────────────────────────────────────────
ticker_counts = df_all.groupby('ticker').size()
valid_tickers = ticker_counts[ticker_counts >= CONFIG['min_history']].index.tolist()
df_filtered = df_all[df_all['ticker'].isin(valid_tickers)].copy()

# ── Primary ticker selection ──────────────────────────────────────────────────
PRIMARY_TICKER = 'FPT'
df = df_filtered[df_filtered['ticker'] == PRIMARY_TICKER].copy().reset_index(drop=True)

# ── Merge all VNI features into the primary ticker DataFrame ──────────────────
df = pd.merge(df, df_vni, on='date', how='left')

vni_feature_cols = [c for c in df_vni.columns if c != 'date']
nan_count = df[vni_feature_cols].isna().sum().sum()
if nan_count > 0:
    print(f'⚠ {nan_count} NaN values after VNI merge — filling with 0')
    df[vni_feature_cols] = df[vni_feature_cols].fillna(0)

print(f'\nPrimary Ticker : {PRIMARY_TICKER} ({len(df):,} rows)')
print(f'VNI features merged: {vni_feature_cols}')




In [ ]:
df['close'] = df['close'].astype(np.float64)

N = CONFIG['n_day']   # forward horizon — driven by CONFIG, not hardcoded
df['target_single'] = np.log(df['close'].shift(-N) / df['close'])   # Task 2.2: log(P_{t+N}/P_t)
df['target_multi']  = np.log(df['close'] / df['close'].shift(1))    # Task 2.3: 1-day log return

print(f'target_single (n={N}): non-null={df["target_single"].notna().sum()}')
print(f'target_multi        : non-null={df["target_multi"].notna().sum()}')

In [ ]:
df_model = df.dropna() 
df_model = df_model.sort_values('date').reset_index(drop=True)
df_model.head(100)
# df vẫn giữ nguyên để tra cứu, df_model dùng để train

---
## Section 3 - Feature Engineering:

In [ ]:
print(f'Columns available: {df_model.columns.tolist()}')
print('-' * 50)

# Exclude metadata, absolute prices (non-stationary), and forward-looking target columns.
# Keeping absolute prices as features causes gradient vanishing / overfitting.
# Target columns must be excluded to prevent data leakage.
COLS_TO_IGNORE = {
    'date', 'ticker',
    'close', 'open', 'high', 'low', 'adj_close',
    'log_return', 'log_return_3d',
    'target_single', 'target_multi',
}

FEATURE_COLS = [c for c in df_model.columns if c not in COLS_TO_IGNORE]

print(f'Feature count : {len(FEATURE_COLS)}')
print(f'Features      : {FEATURE_COLS}')

In [ ]:
CONFIG['n_features'] = len(FEATURE_COLS)

TARGET_SINGLE = 'target_single'   # Task 2.2: forward log-return log(P_{t+N} / P_t)
TARGET_MULTI  = 'target_multi'    # Task 2.3: 1-day log-return log(P_t / P_{t-1})

CONFIG['target_single'] = TARGET_SINGLE
CONFIG['target_multi']  = TARGET_MULTI

print(f'\u2713 n_features      : {CONFIG["n_features"]}')
print(f'\u2713 Task 2.2 target : {TARGET_SINGLE}  (horizon = {CONFIG["n_day"]} days ahead)')
print(f'\u2713 Task 2.3 target : {TARGET_MULTI}  (k = {CONFIG["k_days"]} consecutive days)')

---
## Section 4 - Chronological Train/ Val/ Test Split

### Critical Rule: Never Shuffle Time-Series Data

Random shuffling is correct for i.i.d. datasets (images, NLP) but **catastrophically wrong**
for time series. Shuffling would:

1. **Destroy temporal structure**: the model would see future market state during training.
2. **Cause data leakage**: sliding windows would span train/test boundaries.
3. **Inflate metrics**: test performance would reflect interpolation, not genuine
   out-of-sample forecasting.

**Correct approach**: sort by date → split at chronological cut-points → build
sliding-window sequences *within each split separately* (implemented in Section 6).

In [ ]:
def chronological_split(df, train_ratio, val_ratio):
    """
    Split a time-ordered DataFrame into train / val / test subsets,
    ensuring no temporal overlap between the date ranges.
    """
    n = len(df)

    # Calculate approximate index-based split points
    approx_train_end_idx = int(n * train_ratio)
    approx_val_end_idx   = int(n * (train_ratio + val_ratio))

    # --- Determine the actual end index for the training set ---
    actual_train_end_idx = approx_train_end_idx
    if actual_train_end_idx < n:
        # Get the date of the last element in the approximate train split
        train_split_date_boundary = df.iloc[max(0, approx_train_end_idx - 1)]['date']

        # Find the first index where the date is strictly greater than this boundary date
        mask_after_train = df['date'] > train_split_date_boundary
        if mask_after_train.any():
            actual_train_end_idx = mask_after_train.idxmax()
        else:
            # If no date is strictly greater, it means all remaining data
            # has dates <= train_split_date_boundary. Assign all remaining to train.
            actual_train_end_idx = n

    # --- Determine the actual end index for the validation set ---
    actual_val_end_idx = approx_val_end_idx
    if actual_val_end_idx < n:
        # Get the date of the last element in the approximate val split
        # Ensure we don't try to access an index before actual_train_end_idx
        val_split_date_boundary_idx = max(actual_train_end_idx, approx_val_end_idx - 1)
        if val_split_date_boundary_idx < n:
            val_split_date_boundary = df.iloc[val_split_date_boundary_idx]['date']
            mask_after_val = df['date'] > val_split_date_boundary
            if mask_after_val.any():
                actual_val_end_idx = mask_after_val.idxmax()
            else:
                actual_val_end_idx = n
        else:
            actual_val_end_idx = n

    # Ensure that actual_val_end_idx is always >= actual_train_end_idx
    actual_val_end_idx = max(actual_train_end_idx, actual_val_end_idx)

    train_df = df.iloc[:actual_train_end_idx].copy()
    val_df   = df.iloc[actual_train_end_idx:actual_val_end_idx].copy()
    test_df  = df.iloc[actual_val_end_idx:].copy()

    return train_df, val_df, test_df

# Execute split on the Primary Ticker data
train_df, val_df, test_df = chronological_split(
    df_model, CONFIG['train_ratio'], CONFIG['val_ratio']
)

print(f'Train : {train_df["date"].min().date()} -> {train_df["date"].max().date()} ({len(train_df):,} rows)')
print(f'Val   : {val_df["date"].min().date()} -> {val_df["date"].max().date()} ({len(val_df):,} rows)')
print(f'Test  : {test_df["date"].min().date()} -> {test_df["date"].max().date()} ({len(test_df):,} rows)')

# Sanity check for temporal integrity
assert train_df['date'].max() < val_df['date'].min(), "Temporal overlap between Train and Val!"
assert val_df['date'].max() < test_df['date'].min(), "Temporal overlap between Val and Test!"
print('\n✓ Chronological integrity confirmed.')

In [ ]:
# Fact check the test dataset
print(train_df['target_single'].value_counts().head())
print(val_df['target_single'].value_counts().head())
print(test_df['target_single'].value_counts().head())

print(train_df['target_multi'].value_counts().head())
print(val_df['target_multi'].value_counts().head())
print(test_df['target_multi'].value_counts().head())

In [ ]:
# Kiểm tra xem có bao nhiêu giá trị khác 0 tuyệt đối
non_zero_multi = df[df['target_multi'] != 0]['target_multi']
print(f"Tỷ lệ dữ liệu có biến động: {len(non_zero_multi) / len(df):.2%}")

# Xem thử 5 giá trị nhỏ nhất khác 0 (để check độ phân giải)
print("Các biến động nhỏ nhất được ghi nhận:")
print(non_zero_multi.abs().sort_values().head(5))

In [ ]:
# ── Visualise the three splits on one Log Return chart ────────────────────────
fig, ax = plt.subplots(figsize=(15, 5))

# Use alpha to see density and high volatility spikes better
ax.plot(train_df['date'], train_df['log_return'],
        color='steelblue', label=f'Train ({len(train_df):,} days)', linewidth=0.8, alpha=0.7)
ax.plot(val_df['date'], val_df['log_return'],
        color='darkorange', label=f'Val ({len(val_df):,} days)', linewidth=0.8, alpha=0.8)
ax.plot(test_df['date'], test_df['log_return'],
        color='seagreen', label=f'Test ({len(test_df):,} days)', linewidth=0.8, alpha=0.8)

# Add a horizontal line at 0 for reference
ax.axhline(0, color='red', linestyle='-', alpha=0.3, linewidth=1)

# Borders for splits
for cut in [train_df['date'].max(), val_df['date'].max()]:
    ax.axvline(cut, color='black', linestyle='--', alpha=0.6, linewidth=1.2)

ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.tick_params(axis='x', rotation=45)

ax.set_title(f'{PRIMARY_TICKER} — Chronological Split (Log Returns)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.legend(loc='upper left')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

---
## Section 5 - Normalization:

RobustScaler:

In [ ]:
# ROOT the orginal folder of the project
MODELS_DIR = ROOT / 'models'

# Create a place to store models
MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
from sklearn.preprocessing import RobustScaler
import joblib

# Separate scalers per task: Task 2.2 target is an N-day log return;
# Task 2.3 target is a 1-day log return — different distributions.
feature_scaler   = RobustScaler()
target_scaler_22 = RobustScaler()   # Task 2.2: forward N-day log return
target_scaler_23 = RobustScaler()   # Task 2.3: 1-day log return

# Fit ONLY on training data to prevent leakage into val/test
train_df[FEATURE_COLS]      = feature_scaler.fit_transform(train_df[FEATURE_COLS])
train_df[[TARGET_SINGLE]]   = target_scaler_22.fit_transform(train_df[[TARGET_SINGLE]])
train_df[[TARGET_MULTI]]    = target_scaler_23.fit_transform(train_df[[TARGET_MULTI]])

val_df[FEATURE_COLS]        = feature_scaler.transform(val_df[FEATURE_COLS])
val_df[[TARGET_SINGLE]]     = target_scaler_22.transform(val_df[[TARGET_SINGLE]])
val_df[[TARGET_MULTI]]      = target_scaler_23.transform(val_df[[TARGET_MULTI]])

test_df[FEATURE_COLS]       = feature_scaler.transform(test_df[FEATURE_COLS])
test_df[[TARGET_SINGLE]]    = target_scaler_22.transform(test_df[[TARGET_SINGLE]])
test_df[[TARGET_MULTI]]     = target_scaler_23.transform(test_df[[TARGET_MULTI]])

MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(feature_scaler,   MODELS_DIR / 'vn_feature_scaler.pkl')
joblib.dump(target_scaler_22, MODELS_DIR / 'vn_target_scaler_22.pkl')
joblib.dump(target_scaler_23, MODELS_DIR / 'vn_target_scaler_23.pkl')

print('\u2713 Normalization complete (RobustScaler, no data leakage).')
print(f'  feature_scaler   : {len(FEATURE_COLS)} features')
print(f'  target_scaler_22 : median={target_scaler_22.center_[0]:.6f}  IQR={target_scaler_22.scale_[0]:.6f}')
print(f'  target_scaler_23 : median={target_scaler_23.center_[0]:.6f}  IQR={target_scaler_23.scale_[0]:.6f}')

---
## Section 6 - Sliding Window Sequence Builder

In [ ]:
# ── Task Parameters ──────────────────────────────────────────────────────────
W       = CONFIG['window_size'] # 20
N_DAY   = CONFIG['n_day']       # 3
K_DAYS  = CONFIG['k_days']      # 5 (Khuyên dùng 5 cho VN, nhưng giữ 7 nếu Anh muốn)
K_RANGE = list(range(1, K_DAYS + 1))

# ── Sliding Window Sequence Function ─────────────────────────────────────────
def create_sequences(feature_array, target_array, window_size, forecast_horizon):
    """
    Standard sequence generator tailored for Shifted Targets (VNStock log_return_3d) 
    and Sequential Targets (Task 2.3).
    
    Logic for Task 2.2:
    - Input (X): [i - window_size : i] -> Model observes history up to day i-1.
    - Target (y): target_array[i - 1] -> Extracts volatility from day i-1 to (i-1)+3 
      (Assuming target_array was pre-shifted using shift(-3)).
    """
    X, y = [], []
    n_samples = len(feature_array)

    # 1. Single-step Forecasting (Task 2.2 - n=3)
    if isinstance(forecast_horizon, int): 
        # Iterate i from window_size to n_samples
        # No need to subtract forecast_horizon because the target is already 
        # embedded at index i-1 via the shift(-3) operation.
        for i in range(window_size, n_samples + 1):
            # X contains historical sessions [t-W, ..., t-1]
            X_window = feature_array[i - window_size : i]
            
            # y is the log_return_3d value already pre-computed at day t-1
            y_label = target_array[i - 1]
            
            X.append(X_window)
            y.append(y_label)

    # 2. Multi-step Forecasting (Task 2.3 - k=5)
    elif isinstance(forecast_horizon, list):
        # Note: For Multi-step, we use standard 1-day log returns (no shift).
        # We look ahead from the end of the window to capture the next k days.
        max_k = max(forecast_horizon)
        for i in range(window_size, n_samples - max_k + 1):
            # X contains historical sessions [t-W, ..., t-1]
            X.append(feature_array[i - window_size : i])
            # Extracts k consecutive days immediately following the window [t, ..., t+k-1]
            y.append(target_array[i : i + max_k, 0])
    
    else:
        raise ValueError("forecast_horizon must be an int or a list of ints")
    
    return np.array(X), np.array(y)

    
def build_multi_ticker_sequences(df_split, task_type='single'):
    all_X, all_y = [], []
    
    for ticker, group in df_split.groupby('ticker'):
        feat_arr = group[FEATURE_COLS].values
        
        if task_type == 'single':
            # Task 2.2 dùng cột 3-day return
            tgt_arr = group[['target_single']].values
            X, y = create_sequences(feat_arr, tgt_arr, W, N_DAY)
        else:
            # Task 2.3 dùng cột 1-day return
            tgt_arr = group[['target_multi']].values
            X, y = create_sequences(feat_arr, tgt_arr, W, K_RANGE)
            
        all_X.append(X)
        all_y.append(y)
    return np.concatenate(all_X), np.concatenate(all_y)

# ── Execute Refined Construction ──────────────────────────────────────────────

# ── Execute Refined Construction ──────────────────────────────────────────────

# Task 2.2: Single-step (n=3)
# Removed 'df_vni' as it's already merged inside the split DataFrames
X_train_22, y_train_22 = build_multi_ticker_sequences(train_df, task_type='single')
X_val_22,   y_val_22   = build_multi_ticker_sequences(val_df,   task_type='single')
X_test_22,  y_test_22  = build_multi_ticker_sequences(test_df,  task_type='single')

# Task 2.3: Multi-step (k=5)
X_train_23, y_train_23 = build_multi_ticker_sequences(train_df, task_type='multi')
X_val_23,   y_val_23   = build_multi_ticker_sequences(val_df,   task_type='multi')
X_test_23,  y_test_23  = build_multi_ticker_sequences(test_df,  task_type='multi')
# ── Summary Output ────────────────────────────────────────────────────────────
print(f'✓ Task 2.2 Sequences (n={N_DAY}):')
print(f'  X_train: {X_train_22.shape} | y_train: {y_train_22.shape}')
print(f'✓ Task 2.3 Sequences (k={K_DAYS}):')
print(f'  X_train: {X_train_23.shape} | y_train: {y_train_23.shape}')

In [ ]:
# Task 2.2 Verification
print("--- Task 2.2 Verification (Shifted Logic) ---")
# The first sequence X_test_22[0] ends at index W-1 in the test_df
actual_value_in_df = test_df.iloc[W - 1]['target_single']
generated_value_in_y = y_test_22[0][0] # indexing [0][0] because y is (N, 1)

print(f"Target in Test_DF (index {W-1}): {actual_value_in_df:.6f}")
print(f"Target in y_test_22 array:       {generated_value_in_y:.6f}")

if np.isclose(actual_value_in_df, generated_value_in_y):
    print("✅ SUCCESS: Task 2.2 is correctly aligned with the shifted target.")
else:
    print("❌ ERROR: Task 2.2 misalignment detected!")

In [ ]:
# Task 2.3 Verification
print("\n--- Task 2.3 Verification (Sequential Path) ---")
# The first window X_test_23[0] covers [0:W]. 
# Therefore, y should cover [W:W+K]
actual_values_in_df = test_df.iloc[W : W + K_DAYS]['target_multi'].values
generated_values_in_y = y_test_23[0]

print(f"Target Path in Test_DF (indices {W} to {W+K_DAYS-1}):\n{actual_values_in_df}")
print(f"Target Path in y_test_23 array:\n{generated_values_in_y}")

if np.allclose(actual_values_in_df, generated_values_in_y):
    print("✅ SUCCESS: Task 2.3 is correctly capturing the 5-day sequential path.")
else:
    print("❌ ERROR: Task 2.3 misalignment detected!")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Helper function to flatten 3D sequences into 2D for ML models
def flatten_sequences(X):
    # Reshape from (Samples, Window, Features) to (Samples, Window * Features)
    return X.reshape(X.shape[0], -1)

# Flattening Task 2.2 data
X_train_22_ml = flatten_sequences(X_train_22)
X_val_22_ml   = flatten_sequences(X_val_22)
X_test_22_ml  = flatten_sequences(X_test_22)

# Targets for Task 2.2 are already 2D (Samples, 1), we just need to flatten to 1D
y_train_22_ml = y_train_22.ravel()
y_val_22_ml   = y_val_22.ravel()
y_test_22_ml  = y_test_22.ravel()


# Flattening Task 2.3 data
X_train_23_ml = flatten_sequences(X_train_23)
X_val_23_ml   = flatten_sequences(X_val_23)
X_test_23_ml  = flatten_sequences(X_test_23)



print(f"Machine Learning Input Shape (Task 2.2): {X_train_22_ml.shape}")
print(f"Machine Learning Input Shape (Task 2.3): {X_train_23_ml.shape}")

### Random Forest Regressor

## Rationale: Why Start with Random Forest?

### Performance Benchmark
Establishes a solid performance floor (MAE/RMSE). If a complex Deep Learning
model cannot outperform this baseline, the added complexity is not justified.

### Non-Linearity
Unlike linear models, Random Forest effectively captures non-linear relationships
and interactions between technical indicators without requiring strict statistical
assumptions.

### Robustness
Less sensitive to outliers and requires minimal hyperparameter tuning to produce
reliable results, making it an ideal "sanity check" for the data pipeline.

---

## Feature Importance & Interpretability

### Dimensionality Reduction
With 47+ features and a 20-day look-back window, Random Forest helps identify
signal from noise. Features with near-zero importance can be pruned to streamline
subsequent Deep Learning architectures.

### Time-Step Analysis
By analysing importance across the flattened window, we can determine which
specific days in the look-back period (e.g., *t*−1 vs. *t*−20) carry the most
predictive weight — a critical insight for designing the LSTM input sequence.

### Leakage Detection
Any feature exhibiting disproportionately high importance (e.g., >90%) serves as
a red flag for potential data leakage or look-ahead bias, and warrants immediate
investigation before proceeding to model training.

In [ ]:
# ── Task 2.2 Baseline: Predict Single Day ─────────────────────────────────────
print("Fitting Random Forest for Task 2.2 (n-th day)...")
rf_22 = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
rf_22.fit(X_train_22_ml, y_train_22_ml) # Predict one value

# ── Task 2.3 Baseline: Predict k-consecutive days ────────────────────────────
print("Fitting Random Forest for Task 2.3 (k-consecutive days)...")
rf_23 = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
# Chú ý: Random Forest mặc định hỗ trợ Multi-output, nên ta giữ nguyên y (Samples, k)
rf_23.fit(X_train_23_ml, y_train_23.reshape(y_train_23.shape[0], -1))

# ── Quick Evaluation ──────────────────────────────────────────────────────────
def evaluate_ml(model, X, y_true, task_name="Task"):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"[{task_name}] MAE: {mae:.6f} | RMSE: {rmse:.6f}")
    return y_pred



In [ ]:

y_pred_22_rf = evaluate_ml(rf_22, X_test_22_ml, y_test_22_ml, "Baseline RF 2.2")
y_pred_23_rf = evaluate_ml(rf_23, X_test_23_ml, y_test_23.reshape(y_test_23.shape[0], -1), "Baseline RF 2.3")

In [ ]:
# Direction
actual_dir = np.sign(y_test_22_ml)
pred_dir = np.sign(y_pred_22_rf)
da_22 = np.mean(actual_dir == pred_dir)
print(f"Directional Accuracy for RF (Task 2.2): {da_22:.2%}")

In [ ]:
def calculate_pointwise_da(y_true, y_pred, task_name="Task 2.3"):
    """
    Calculates Directional Accuracy for each step in a multi-output forecast.
    
    Args:
        y_true (np.ndarray): Actual log returns, shape (Samples, K_days)
        y_pred (np.ndarray): Predicted log returns, shape (Samples, K_days)
    """
    # 1. Get the signs (Direction: -1 for Down, 0 for Flat, 1 for Up)
    actual_signs = np.sign(y_true)
    pred_signs = np.sign(y_pred)
    
    # 2. Calculate accuracy per day (Point-wise)
    # We compare the sign matrices element-wise
    hits = (actual_signs == pred_signs).astype(int)
    
    # Calculate mean across the sample dimension (axis 0)
    da_per_day = np.mean(hits, axis=0) * 100
    
    # 3. Print a Leaderboard style report
    print(f"--- {task_name} Point-wise Directional Accuracy RF ---")
    for i, accuracy in enumerate(da_per_day):
        print(f" Day {i+1} Ahead: {accuracy:.2f}%")
    # Calculate Overall Average
    avg_da = np.mean(da_per_day)
    print(f"---------------------------------------------")
    print(f" Average DA: {avg_da:.2f}%")
    
    return da_per_day

# --- Execution ---
# Flatten y_test if it's 3D, ensure it's (Samples, K_days)
y_true_multi = y_test_23.reshape(y_test_23.shape[0], -1)
# y_pred_23 is the output from your model.predict()
da_results = calculate_pointwise_da(y_true_multi, y_pred_23_rf)

In [ ]:
from sklearn.linear_model import LinearRegression

# ── Task 2.2 Baseline: Linear Regression ──────────────────────────────────────
print("Training Linear Regression for Task 2.2...")
lr_22 = LinearRegression()
lr_22.fit(X_train_22_ml, y_train_22_ml)

# ── Task 2.3 Baseline: Linear Regression ──────────────────────────────────────
# Scikit-learn LR also supports multi-output (N, k)
print("Training Linear Regression for Task 2.3...")
lr_23 = LinearRegression()
lr_23.fit(X_train_23_ml, y_train_23.reshape(y_train_23.shape[0], -1))

def evaluate_simple(model, X, y_true, name):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{name:25} | MAE: {mae:.6f}")
    return y_pred

y_pred_lr22 = evaluate_simple(lr_22, X_test_22_ml, y_test_22_ml, "Linear Regression 2.2")
y_pred_lr23 = evaluate_simple(lr_23, X_test_23_ml, y_test_23.reshape(y_test_23.shape[0], -1), "Linear Regression 2.3")

In [ ]:
!pip install xgboost

In [ ]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
import numpy as np

# ── Task 2.2: XGBoost (Single Output) ─────────────────────────────────────────
print("Training XGBoost for Task 2.2...")
xgb_22 = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    tree_method='hist', 
    random_state=42,
    n_jobs=-1
)
# Đảm bảo target là 1D cho single output
xgb_22.fit(X_train_22_ml, y_train_22_ml)

# ── Task 2.3: XGBoost (Multi-output) ─────────────────────────────────────────
print("Training XGBoost for Task 2.3 (Multi-output)...")
xgb_base = xgb.XGBRegressor(
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.05, 
    tree_method='hist',
    random_state=42
)

# Wrapper để XGBoost có thể dự báo cùng lúc k ngày (k=5)
xgb_23 = MultiOutputRegressor(xgb_base)

# Cần reshape y_train_23 từ (samples, 5, 1) thành (samples, 5)
y_train_23_reshaped = y_train_23.reshape(y_train_23.shape[0], -1)
xgb_23.fit(X_train_23_ml, y_train_23_reshaped)

# ── Đánh giá nhanh ────────────────────────────────────────────────────────────
y_pred_xgb22 = evaluate_simple(xgb_22, X_test_22_ml, y_test_22_ml, "XGBoost 2.2")
y_pred_xgb23 = evaluate_simple(xgb_23, X_test_23_ml, y_test_23.reshape(y_test_23.shape[0], -1), "XGBoost 2.3")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------------
# 1. Create Feature Names for the Flattened Input
# -------------------------------------------------------------------
# X_train shape is (Samples, Window_size, Features)
# After flattening, features are ordered: Day 1 (all features) -> Day 2 -> ... -> Day 20
flattened_feature_names = []
W = CONFIG['window_size']

for day in range(1, W + 1):
    for feat in FEATURE_COLS:
        # Naming convention: "t-19_rsi", "t-18_rsi", ..., "t_rsi" (current day)
        days_ago = W - day 
        if days_ago == 0:
            flattened_feature_names.append(f"t_{feat}")
        else:
            flattened_feature_names.append(f"t-{days_ago}_{feat}")

# Extract importance scores from the XGBoost Task 2.2 model
importances = xgb_22.feature_importances_

# -------------------------------------------------------------------
# 2. Structure Importance Data into a DataFrame
# -------------------------------------------------------------------
df_importance = pd.DataFrame({
    'Feature': flattened_feature_names,
    'Importance': importances
})

# Create a 'Base_Feature' column by stripping the time prefix (e.g., "t-19_rsi" -> "rsi")
# This allows us to see the overall impact of a feature regardless of the specific day
df_importance['Base_Feature'] = df_importance['Feature'].apply(
    lambda x: x.split('_', 1)[1] if '_' in x else x
)

# Sort by importance in descending order
df_importance = df_importance.sort_values(by='Importance', ascending=False)

# -------------------------------------------------------------------
# Visualization 1: Top 20 Specific Time-Step Features
# -------------------------------------------------------------------
plt.figure(figsize=(12, 8))
sns.barplot(
    data=df_importance.head(20), 
    x='Importance', 
    y='Feature', 
    palette='viridis'
)
plt.title('Top 20 Most Important Flattened Features (XGBoost Task 2.2)', fontsize=14)
plt.xlabel('Importance Score (Gain)')
plt.ylabel('Feature (Time-Step_Name)')
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# Visualization 2: Aggregated Importance by Base Feature
# (Reveals which data categories are actually driving the model)
# -------------------------------------------------------------------
df_aggregated = df_importance.groupby('Base_Feature')['Importance'].sum().reset_index()
df_aggregated = df_aggregated.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 10))
sns.barplot(
    data=df_aggregated.head(25), # Show Top 25 base feature groups
    x='Importance', 
    y='Base_Feature', 
    palette='magma'
)
plt.title('Aggregated Feature Importance (Sum over 20-day window)', fontsize=14)
plt.xlabel('Total Importance Score')
plt.ylabel('Base Feature')
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 3. Suggestions for Dimensionality Reduction
# -------------------------------------------------------------------
# Identify features with near-zero importance
useless_features = df_aggregated[df_aggregated['Importance'] <= 0.0001]['Base_Feature'].tolist()

print("-" * 30)
print(f"IDENTIFIED {len(useless_features)} BASE FEATURES WITH NEAR-ZERO IMPORTANCE:")
if useless_features:
    print(useless_features)
    print("\nRecommendation: Consider removing these to reduce noise and improve Deep Learning performance.")
else:
    print("None. All features contribute some signal to the model.")
print("-" * 30)

#### Evaluate

In [ ]:
def plot_prediction_analysis(y_true, y_pred, title="Model Evaluation"):
    # Chuyển ngược từ Scaled về Original Log Return (nếu cần)
    # y_true_orig = target_scaler.inverse_transform(y_true)
    # y_pred_orig = target_scaler.inverse_transform(y_pred)

    plt.figure(figsize=(15, 6))
    plt.plot(y_true[-100:], label='Actual Log Return', color='black', linewidth=1.5, alpha=0.7)
    plt.plot(y_pred[-100:], label='Predicted Log Return', color='red', linestyle='--', linewidth=1.5)

    plt.title(f'{title} - Last 100 Days Analysis', fontsize=14)
    plt.xlabel('Time Steps (Test Set)')
    plt.ylabel('Log Return')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# Sử dụng cho XGBoost
plot_prediction_analysis(y_test_22, y_pred_xgb22, "XGBoost Task 2.2")

In [ ]:
errors = y_test_22.ravel() - y_pred_xgb22.ravel()
plt.figure(figsize=(8, 5))
plt.hist(errors, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--')
plt.title('Error Distribution (Residuals) for XGBoost Task 2.2')
plt.show()

In [ ]:
def plot_multi_step_path(y_true, y_pred, sample_idx):
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, K_DAYS + 1), y_true[sample_idx], marker='o', label='Actual Path', color='black')
    plt.plot(range(1, K_DAYS + 1), y_pred[sample_idx], marker='s', label='Predicted Path', color='red', linestyle='--')

    plt.title(f'Multi-Day Forecast for Task 2.3 (Starting from Sample {sample_idx})')
    plt.xlabel('Days Ahead')
    plt.ylabel('Log Return 3D')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# Vẽ thử một mẫu bất kỳ cho Task 2.3
plot_multi_step_path(y_test_23.reshape(y_test_23.shape[0], -1), y_pred_xgb23, sample_idx=50)

In [ ]:
def get_predictions_vn(model, X_input, test_df_raw, target_scaler, window_size, forecast_horizon):
    ""
    # 1. Inference — handles both sklearn (no verbose kwarg) and Keras
    try:
        y_pred_scaled = model.predict(X_input, verbose=0)
    except TypeError:
        y_pred_scaled = model.predict(X_input)

    n           = y_pred_scaled.shape[0]
    start_idx   = window_size - 1
    base_prices = test_df_raw['close'].values[start_idx : start_idx + n]

    if isinstance(forecast_horizon, int):
        # Task 2.2: single-step — inverse-scale (n,1), then P_{t+N} = P_t * exp(r)
        raw_log = target_scaler.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).flatten()
        return base_prices * np.exp(raw_log)
    else:
        # Task 2.3: multi-step.
        # Scaler was fitted on 1 column, so reshape (n,k) -> (n*k,1) -> inverse -> (n,k).
        k       = len(forecast_horizon)
        raw_log = target_scaler.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).reshape(n, k)

        price_path = np.zeros_like(raw_log)
        price_path[:, 0] = base_prices * np.exp(raw_log[:, 0])
        for i in range(1, k):
            price_path[:, i] = price_path[:, i - 1] * np.exp(raw_log[:, i])
        return price_path

In [ ]:
# Predict prices for Task 2.2 (XGBoost)
y_pred_price_22 = get_predictions_vn(xgb_22, X_test_22, test_df, target_scaler_22, W, N_DAY)

# Actual price at t+N: first window ends at index W-1, target price is at W-1+N
y_true_price_22 = test_df['close'].values[W + N_DAY - 1 : W + N_DAY - 1 + len(y_pred_price_22)]

print(f'Sample Pred Price: {y_pred_price_22[0]:.2f}')
print(f'Sample True Price: {y_true_price_22[0]:.2f}')

In [ ]:
# ── Price-level evaluation for Task 2.2 ──────────────────────────────────────

# 1. Inverse-scale to raw log returns (target_scaler_22 fitted on target_single)
y_test_orig = target_scaler_22.inverse_transform(y_test_22.reshape(-1, 1)).flatten()
y_pred_orig = target_scaler_22.inverse_transform(y_pred_xgb22.reshape(-1, 1)).flatten()

# 2. Base close price at end of each input window (day t)
W     = CONFIG['window_size']
N_DAY = CONFIG['n_day']
current_close = test_df['close'].values[W - 1 : len(test_df) - N_DAY]

# Align lengths (shift(-N) drops N rows from the tail)
min_len = min(len(y_test_orig), len(current_close))
y_test_orig, y_pred_orig, current_close = (
    y_test_orig[:min_len], y_pred_orig[:min_len], current_close[:min_len]
)

# 3. Reconstruct prices: P_{t+N} = P_t * exp(log_return_N)
price_actual = current_close * np.exp(y_test_orig)
price_pred   = current_close * np.exp(y_pred_orig)

# 4. Metrics
mae_price  = np.mean(np.abs(price_actual - price_pred))
actual_dir = np.sign(y_test_orig)
pred_dir   = np.sign(y_pred_orig)
da_score   = np.mean(actual_dir == pred_dir)

print(f'\u2713 Price MAE  (N={N_DAY}d): {mae_price:.2f}')
print(f'\u2713 Directional Accuracy : {da_score:.2%}')

---
## Section - Build Deep Learning models

In [ ]:
def build_seq2seq_attention(window_size, n_features, output_steps, name='seq2seq_attention'):
    """
    Encoder-Decoder GRU với Multi-Head Attention.
    Thích hợp nhất cho Task 1.3: Dự báo quỹ đạo 5-7 ngày.
    """
    # --- ENCODER ---
    inputs = tf.keras.Input(shape=(window_size, n_features), name='input')

    # GRU trích xuất đặc trưng chuỗi
    encoder_seq, encoder_state = tf.keras.layers.GRU(
        128, return_sequences=True, return_state=True, name='encoder_gru')(inputs)

    # Attention Layer giúp Encoder tập trung vào các phiên quan trọng trong quá khứ
    attention = tf.keras.layers.MultiHeadAttention(
        num_heads=4, key_dim=32, name='attention')(encoder_seq, encoder_seq)

    # Tổng hợp thông tin từ Attention
    context_vector = tf.keras.layers.GlobalAveragePooling1D()(attention)

    # --- BRIDGE ---
    # Lặp lại vector ngữ cảnh cho k ngày dự báo
    x = tf.keras.layers.RepeatVector(output_steps, name='bridge')(context_vector)

    # --- DECODER ---
    # Dùng encoder_state làm "mầm" cho bộ nhớ của Decoder
    x = tf.keras.layers.GRU(
        128, return_sequences=True, name='decoder_gru')(x, initial_state=encoder_state)

    x = tf.keras.layers.Dropout(0.2, name='dropout')(x)

    # Dự báo giá trị cho từng ngày trong chuỗi k ngày
    outputs = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(1), name='time_dist_dense')(x)

    # Reshape (batch, k, 1) -> (batch, k)
    outputs = tf.keras.layers.Reshape((output_steps,), name='output')(outputs)

    model = tf.keras.Model(inputs, outputs, name=name)
    return model

# ── Initialization ────────────────────────────────────────────────────────────
model_s2s_attn = build_seq2seq_attention(
    window_size=CONFIG['window_size'],
    n_features=CONFIG['n_features'],
    output_steps=CONFIG['k_days']
)
model_s2s_attn.summary()

In [ ]:
def compile_and_train_vn(model, X_tr, y_tr, X_va, y_va, save_name):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
        loss='mse',
        metrics=['mae']
    )

    save_path = MODELS_DIR / f'{save_name}.keras'

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=CONFIG['patience'], restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            str(save_path), monitor='val_loss', save_best_only=True, verbose=0)
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=CONFIG['epochs'],
        batch_size=CONFIG['batch_size'],
        callbacks=callbacks,
        verbose=1
    )
    return history

# ── Run Training for Task 2.3 ─────────────────────────────────────────────────
history_s2s = compile_and_train_vn(
    model_s2s_attn, X_train_23, y_train_23, X_val_23, y_val_23, "s2s_attn_vn_task23"
)